# 1. Настройка окружения

## 1.1 Установка версий torch, torchvision, torchaudio,совместимых с SAM3

In [1]:
%%capture
!pip uninstall torch torchvision torchaudio -y 2>/dev/null
!pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu126

## 1.2 Монтирование Google Drive

In [33]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import warnings
warnings.filterwarnings('ignore')

import os
import sys

Mounted at /content/drive


## 1.3 Создание папок и установка всех зависимостей

In [34]:
# Создаем структуру папок
!mkdir -p /content/drive/MyDrive/sam3_bdd_project/checkpoints
!mkdir -p /content/drive/MyDrive/sam3_bdd_project/data_seg
!mkdir -p /content/drive/MyDrive/sam3_bdd_project/results

# Создаем символические ссылки для удобства
!ln -sfn /content/drive/MyDrive/sam3_bdd_project/checkpoints /content/checkpoints
!ln -sfn /content/drive/MyDrive/sam3_bdd_project/data_seg /content/data
!ln -sfn /content/drive/MyDrive/sam3_bdd_project/results /content/results

# Установка YOLOv8
!pip install ultralytics -q

!apt-get update -qq
!apt-get install -y ninja-build gcc g++ -qq
!pip install -U pip setuptools wheel -q
!pip install openmim -q
!mim install mmcv==2.1.0 -q

# Установка остальных зависимостей
!pip install timm ftfy iopath hydra-core submitit fvcore fairscale yacs -q
!pip install decord mmengine segment-anything addict yapf portalocker -q

# Установка EfficientSAM3 из официального репозитория
!git clone https://github.com/SimonZeng7108/efficientsam3.git
%cd efficientsam3
!pip install -e . -q
%cd ..

# Установка дополнительных зависимостей
!pip install scipy opencv-python pillow matplotlib -q

# Вариант ES-TV-S (TinyViT-5M) - рекомендуемый для баланса
if not os.path.exists('/content/checkpoints/efficient_sam3_tinyvit_s.pt'):
    print("📥 Скачиваем EfficientSAM3 TinyViT-5M веса...")
    !wget https://huggingface.co/Simon7108528/EfficientSAM3/resolve/main/stage1_all_converted/efficient_sam3_tinyvit_s.pt \
        -O /content/checkpoints/efficient_sam3_tinyvit_s.pt
    print("✅ EfficientSAM3 weights downloaded")
else:
    print("✅ EfficientSAM3 weights already exist")

# Скачивание весов YOLO
if not os.path.exists('/content/checkpoints/yolov8n.pt'):
    print("📥 Скачиваем YOLO веса...")
    !wget https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt \
        -O /content/checkpoints/yolov8n.pt
    print("YOLO weights downloaded")
else:
    print("YOLO weights already exist")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
openxlab 0.1.3 requires setuptools~=60.2.0, but you have setuptools 82.0.1 which is incompatible.
pytensor 2.38.2 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.2 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
Traceback (most recent call last):
  File "/usr/local/bin

## 1.4 Импорт библиотек

In [9]:
import time
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2
from typing import List, Dict, Tuple, Optional
from enum import Enum
from skimage.feature import hog
import json
import torch
import torchvision
import torchaudio
from ultralytics import YOLO
from typing import Dict, List, Tuple, Optional
import glob
import shutil
import gc
from scipy.optimize import linear_sum_assignment
from scipy import ndimage
sys.path.append('/content/efficientsam3')
from sam3.model_builder import build_efficientsam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

## 1.5 Проверка настройки окружения

In [10]:
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")

PyTorch: 2.7.0+cu126
TorchVision: 0.22.0+cu126


In [11]:
# Проверка импортов
print("\nПроверка установки...")

try:
    from ultralytics import YOLO
    print("YOLO imported successfully")
except ImportError as e:
    print(f"!!!YOLO import error: {e}")

try:
    from sam3.model_builder import build_efficientsam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor
    print("EfficientSAM3 imported successfully")
except ImportError as e:
    print(f"!!!EfficientSAM3 import error: {e}")
    print("Проверьте установку: cd /content/efficientsam3 && pip install -e .")


Проверка установки...
YOLO imported successfully
EfficientSAM3 imported successfully


# 2. Основной пайплайн, YOLO+SAM3 из коробки

## 2.1 Класс определения условий видимости

In [12]:
class VisibilityCondition(Enum):
    NIGHT = "night"
    RAIN = "rain"
    SNOW = "snow"
    FOG = "fog"
    CLEAR = "clear"


def compute_hog_features(gray: np.ndarray) -> np.ndarray:
    """Вычисление HOG-признаков для анализа текстуры"""
    features = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                   cells_per_block=(2, 2), visualize=False)
    return features


def detect_visibility_condition(image: np.ndarray) -> VisibilityCondition:
    """
    Определяет тип условий видимости по характеристикам изображения.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # 1. Средняя яркость (для определения ночи)
    mean_brightness = gray.mean()

    # 2. Контраст (для определения тумана)
    contrast = gray.std()

    # 3. Высокочастотные компоненты (для дождя/снега)
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    high_freq_energy = laplacian.var()

    # 4. HOG-признаки (для тумана)
    hog_features = compute_hog_features(gray)
    hog_sparsity = np.mean(hog_features < 0.01)

    # 5. Горизонтальные и вертикальные градиенты (для дождя/снега)
    horizontal_edges = cv2.Sobel(gray, cv2.CV_64F, 1, 0)   # ← ДОБАВЛЕНО: dx=1, dy=0
    vertical_edges = cv2.Sobel(gray, cv2.CV_64F, 0, 1)     # dx=0, dy=1

    # Классификация условий
    if mean_brightness < 60:
        return VisibilityCondition.NIGHT
    elif contrast < 40 and hog_sparsity > 0.7:
        return VisibilityCondition.FOG
    elif high_freq_energy > 800:
        # Дождь → вертикальные градиенты преобладают (капли падают вниз)
        if vertical_edges.var() > horizontal_edges.var() * 1.5:
            return VisibilityCondition.RAIN
        else:
            return VisibilityCondition.SNOW
    else:
        return VisibilityCondition.CLEAR

## 2.1 Вспомогательные функции

In [13]:
def clear_gpu_memory():
    """Очистка памяти GPU"""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        gc.collect()

In [14]:
def convert_color_mask_to_trainid(color_mask: np.ndarray) -> np.ndarray:
    h, w = color_mask.shape[:2]
    trainid_mask = np.zeros((h, w), dtype=np.uint8)
    for rgb, train_id in BDD100K_COLOR_TO_TRAINID.items():
        trainid_mask[np.all(color_mask == rgb, axis=-1)] = train_id
    return trainid_mask

In [15]:
def compute_iou(mask_pred: np.ndarray, mask_gt: np.ndarray) -> float:
    intersection = np.logical_and(mask_pred, mask_gt).sum()
    union = np.logical_or(mask_pred, mask_gt).sum()
    return float(intersection) / float(union) if union > 0 else 0.0

In [16]:
def load_gt_mask(mask_path: str, target_size: Tuple[int, int]) -> np.ndarray:
    gt_mask = Image.open(mask_path).resize(target_size, Image.NEAREST)
    gt_array = np.array(gt_mask)
    if gt_array.ndim == 3 and gt_array.shape[2] >= 3:
        gt_array = convert_color_mask_to_trainid(gt_array[:, :, :3])
    elif gt_array.ndim == 3 and gt_array.shape[2] == 1:
        gt_array = gt_array[:, :, 0]
    return gt_array.astype(np.uint8)

In [17]:
def get_instance_masks_from_gt(gt_mask: np.ndarray, target_classes: List[str]) -> Dict[str, List[np.ndarray]]:
    instance_masks = {}
    class_to_trainid = {v: k for k, v in BDD100K_TRAINID_TO_CLASS.items()}

    for cls in target_classes:
        train_ids = [tid for tid, name in BDD100K_TRAINID_TO_CLASS.items() if name == cls]
        all_masks = []

        for train_id in train_ids:
            class_mask = (gt_mask == train_id)
            if not class_mask.any():
                continue

            if cls in ['person', 'bicycle', 'motorcycle']:
                class_mask = ndimage.binary_closing(class_mask, structure=np.ones((3, 3), dtype=bool))

            labeled, num_components = ndimage.label(class_mask)
            min_area = CLASS_SPECIFIC_MIN_AREA.get(cls, 100)

            for i in range(1, num_components + 1):
                instance_mask = (labeled == i)
                if instance_mask.sum() >= min_area:
                    all_masks.append(instance_mask.astype(np.uint8))

        instance_masks[cls] = all_masks

    return instance_masks

In [18]:
def match_predictions_to_gt(
    pred_masks: List[np.ndarray],
    pred_classes: List[str],
    gt_masks_per_class: Dict[str, List[np.ndarray]],
) -> Tuple[List[float], List[Optional[int]], List[Optional[str]]]:
    if not pred_masks:
        return [], [], []

    n_pred = len(pred_masks)
    ious = np.zeros(n_pred)
    matched_indices = [None] * n_pred
    matched_classes = [None] * n_pred

    pred_by_class = {}
    for i, (mask, cls) in enumerate(zip(pred_masks, pred_classes)):
        pred_by_class.setdefault(cls, []).append(i)

    for cls, pred_indices in pred_by_class.items():
        gt_masks = gt_masks_per_class.get(cls, [])
        if not gt_masks:
            continue

        n_pred_cls, n_gt_cls = len(pred_indices), len(gt_masks)
        cost_matrix = np.zeros((n_pred_cls, n_gt_cls))

        for i, pred_idx in enumerate(pred_indices):
            for j, gt_mask in enumerate(gt_masks):
                cost_matrix[i, j] = 1 - compute_iou(pred_masks[pred_idx], gt_mask)

        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        for r, c in zip(row_ind, col_ind):
            if r < n_pred_cls and c < n_gt_cls:
                pred_idx = pred_indices[r]
                ious[pred_idx] = 1 - cost_matrix[r, c]
                matched_indices[pred_idx] = c
                matched_classes[pred_idx] = cls

    return ious.tolist(), matched_indices, matched_classes

In [19]:
def compute_class_level_iou(pred_masks, pred_classes, gt_mask, target_classes):
    """
    Вычисляет IoU на уровне классов:
    объединяет все предсказанные маски одного класса и сравнивает с GT.
    """
    class_to_trainid = {v: k for k, v in BDD100K_TRAINID_TO_CLASS.items()}

    class_ious = {}
    for cls in target_classes:
        # Объединяем все предсказанные маски этого класса
        pred_union = np.zeros(gt_mask.shape, dtype=bool)
        for mask, pc in zip(pred_masks, pred_classes):
            if pc == cls and mask.shape == gt_mask.shape:
                pred_union = np.logical_or(pred_union, mask)

        # GT маска для этого класса
        train_ids = [tid for tid, name in BDD100K_TRAINID_TO_CLASS.items() if name == cls]
        gt_class = np.zeros(gt_mask.shape, dtype=bool)
        for tid in train_ids:
            gt_class = np.logical_or(gt_class, gt_mask == tid)

        # IoU
        if pred_union.any() or gt_class.any():
            class_ious[cls] = compute_iou(pred_union.astype(np.uint8), gt_class.astype(np.uint8))
        else:
            class_ious[cls] = 1.0  # Оба пустые — идеальное совпадение

    overall_iou = np.mean(list(class_ious.values()))
    return class_ious, overall_iou

In [20]:
def compute_iou_for_pipeline(pred_masks, pred_classes, gt_mask) -> Dict:
    gt_instances = get_instance_masks_from_gt(gt_mask, TARGET_CLASSES)
    ious, matched_indices, matched_classes = match_predictions_to_gt(pred_masks, pred_classes, gt_instances)

    per_class_iou, per_class_counts = {}, {}
    for cls in TARGET_CLASSES:
        cls_ious = [iou for iou, pc in zip(ious, pred_classes) if pc == cls]
        per_class_iou[cls] = np.mean(cls_ious) if cls_ious else 0.0
        per_class_counts[cls] = len(cls_ious)

    valid_ious = [iou for iou, pc in zip(ious, pred_classes) if pc in TARGET_CLASSES]
    miou = np.mean(valid_ious) if valid_ious else 0.0

    return {
        'per_class_iou': per_class_iou,
        'per_class_counts': per_class_counts,
        'miou': miou,
        'ious': ious,
        'matched_indices': matched_indices,
        'matched_classes': matched_classes,
        'total_predictions': len(ious),
        'matched_predictions': sum(1 for idx in matched_indices if idx is not None)
    }

In [21]:
def enhance_night(image: np.ndarray) -> np.ndarray:
    """Быстрая обработка ночи: баланс белого + CLAHE"""
    # 1. Исправить цветовую температуру (желтизна фонарей)
    mean_rgb = image.mean(axis=(0, 1))
    gray_value = mean_rgb.mean()
    scale = gray_value / mean_rgb
    image = np.clip(image * scale, 0, 255).astype(np.uint8)

    # 2. Адаптивная яркость: сильнее в тёмных областях
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # Два прохода CLAHE: для теней и для света
    clahe_dark = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4,4))
    clahe_bright = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8))

    l_dark = clahe_dark.apply(l)
    l_bright = clahe_bright.apply(l)

    # Маска: где темно — сильнее осветляем
    mask = (l < 50).astype(np.float32)
    l_enhanced = (l_dark * mask + l_bright * (1 - mask)).astype(np.uint8)

    return cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2RGB)


def enhance_rain(image: np.ndarray) -> np.ndarray:
    """
    Быстрая предобработка дождя.
    Медианный фильтр + лёгкое повышение контраста.
    """
    # 1. Медианный фильтр (ядро 5×5)
    denoised = cv2.medianBlur(image, 5)

    # 2. Смешивание с оригиналом (70/30)
    enhanced = cv2.addWeighted(image, 0.7, denoised, 0.3, 0)

    # 3. Лёгкий CLAHE для контраста
    lab = cv2.cvtColor(enhanced, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8,8))
    l = clahe.apply(l)

    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def enhance_snow(image: np.ndarray) -> np.ndarray:
    """
    Удаление снежного шума с сохранением границ объектов.
    """
    # Билатеральный фильтр
    denoised = cv2.bilateralFilter(image, d=9, sigmaColor=75, sigmaSpace=7)

    # CLAHE для восстановления контраста
    lab = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l_enhanced = clahe.apply(l)

    return cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2RGB)


def guided_filter(guide: np.ndarray, src: np.ndarray, radius: int, eps: float) -> np.ndarray:
    """Guided Filter — сохраняет границы при фильтрации."""
    mean_guide = cv2.boxFilter(guide, cv2.CV_64F, (radius, radius))
    mean_src = cv2.boxFilter(src, cv2.CV_64F, (radius, radius))

    corr_guide = cv2.boxFilter(guide * guide, cv2.CV_64F, (radius, radius))
    corr_guide_src = cv2.boxFilter(guide * src, cv2.CV_64F, (radius, radius))

    var_guide = corr_guide - mean_guide * mean_guide
    cov_guide_src = corr_guide_src - mean_guide * mean_src

    a = cov_guide_src / (var_guide + eps)
    b = mean_src - a * mean_guide

    mean_a = cv2.boxFilter(a, cv2.CV_64F, (radius, radius))
    mean_b = cv2.boxFilter(b, cv2.CV_64F, (radius, radius))

    return mean_a * guide + mean_b


def enhance_fog_dcp(image: np.ndarray, omega: float = 0.85, t0: float = 0.1) -> np.ndarray:
    """
    Удаление тумана через Dark Channel Prior.
    """
    from scipy import ndimage as ndi

    img_float = image.astype(np.float64) / 255.0

    # Dark Channel
    dark_channel = ndi.minimum_filter(img_float.min(axis=2), size=15)

    # Atmospheric light
    num_pixels = dark_channel.size // 1000
    flat_img = img_float.reshape(-1, 3)
    flat_dark = dark_channel.reshape(-1)
    indices = np.argpartition(flat_dark, -num_pixels)[-num_pixels:]
    atmospheric_light = np.max(flat_img[indices], axis=0)

    # Transmission map
    transmission = 1 - omega * dark_channel
    transmission = guided_filter(img_float.mean(axis=2), transmission, radius=40, eps=1e-6)
    transmission = np.maximum(transmission, t0)

    # Recovery
    enhanced = np.zeros_like(img_float)
    for c in range(3):
        enhanced[:,:,c] = (img_float[:,:,c] - atmospheric_light[c]) / transmission + atmospheric_light[c]

    enhanced = np.clip(enhanced, 0, 1)
    return (enhanced * 255).astype(np.uint8)

In [22]:
def enhance_clahe(image: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_enhanced = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2RGB)

In [23]:
def denoise_nlm(image: np.ndarray) -> np.ndarray:
    return cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21)

In [24]:
def unsharp_mask(image: np.ndarray, strength: float = 1.5) -> np.ndarray:
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=3)
    return cv2.addWeighted(image, strength, blurred, 1 - strength, 0)

In [25]:
def gray_world_white_balance(image: np.ndarray) -> np.ndarray:
    mean_rgb = image.mean(axis=(0, 1))
    gray_value = mean_rgb.mean()
    scale = gray_value / mean_rgb
    return np.clip(image * scale, 0, 255).astype(np.uint8)

In [26]:
def close_mask_holes(mask: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    closed = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    return closed.astype(bool)

In [27]:
def preprocess_for_visibility(image: np.ndarray,
                             condition: VisibilityCondition = None) -> np.ndarray:
    """
    Унифицированная предобработка под условия видимости.
    Каждый метод оптимизирован под конкретную погоду.
    """
    if condition is None:
        condition = detect_visibility_condition(image)

    if condition == VisibilityCondition.NIGHT:
        return enhance_night(image)

    elif condition == VisibilityCondition.RAIN:
        return enhance_rain(image)

    elif condition == VisibilityCondition.SNOW:
        return enhance_snow(image)

    elif condition == VisibilityCondition.FOG:
        return enhance_fog_dcp(image)

    else:
        return image

## 2.3 Основной класс

In [38]:
class YOLO_SAM_Pipeline_Baseline:
    """YOLO26n + EfficientSAM3 — базовая версия без улучшений (baseline)"""

    def __init__(self,
                 sam_checkpoint='/content/checkpoints/efficient_sam3_efficientvit_s.pt',
                 confidence_threshold=0.5,
                 backbone_type='efficientvit',
                 model_name='b0',
                 cache_dir: str = '/content/image_cache'):

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"  Device: {self.device}")

        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

        # Загрузка YOLO
        print(" Loading YOLO26n...")
        self.yolo = YOLO('yolo26n.pt').to(self.device)
        self.yolo.model.half()
        self.confidence_threshold = confidence_threshold

        # Загрузка SAM
        print(" Loading EfficientSAM3...")
        self.sam = build_efficientsam3_image_model(
            checkpoint_path=sam_checkpoint,
            backbone_type=backbone_type,
            model_name=model_name,
            enable_inst_interactivity=True,
        ).to(self.device).eval()

        self.processor = Sam3Processor(self.sam)

        self.class_names = {
            0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle',
            5: 'bus', 7: 'truck', 9: 'traffic light'
        }

        print(f" Baseline Pipeline ready (conf={self.confidence_threshold})\n")

    # вспомогательные методы
    def _get_cached_path(self, remote_path: str) -> str:
        filename = Path(remote_path).name
        local_path = os.path.join(self.cache_dir, filename)
        if not os.path.exists(local_path):
            shutil.copy2(remote_path, local_path)
        return local_path

    def preload_images(self, image_paths: List[str]) -> None:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=4) as executor:
            list(executor.map(self._get_cached_path, image_paths))

    def _process_mask(self, mask):
        mask = np.squeeze(mask)
        if mask.ndim == 3:
            mask = mask[0] if mask.shape[0] == 3 else mask[:, :, 0]
        if mask.dtype == bool:
            return mask.astype(np.uint8)
        elif mask.dtype in [np.float32, np.float64, float]:
            return (mask > 0.5).astype(np.uint8)
        else:
            return (mask > 127).astype(np.uint8)

    def _resize_mask(self, mask, target_size):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
        mask_resized = mask_pil.resize(target_size, Image.NEAREST)
        return (np.array(mask_resized) > 127).astype(np.uint8)

    def _process_sam_result(self, result, num_objects, original_size, yolo_confs):
        masks_list, scores_list = [], []

        if isinstance(result, (tuple, list)) and len(result) >= 2:
            all_masks, all_scores = result[0], result[1]
        else:
            all_masks, all_scores = result, None

        if all_masks is None:
            return [], []

        if isinstance(all_masks, torch.Tensor):
            all_masks = all_masks.cpu().numpy()
        if isinstance(all_scores, torch.Tensor):
            all_scores = all_scores.cpu().numpy()

        num_masks_per_obj = all_masks.shape[1] if isinstance(all_masks, np.ndarray) and all_masks.ndim == 4 else 1

        for i in range(num_objects):
            best_mask, best_score = None, 0.0
            for j in range(num_masks_per_obj):
                mask = all_masks[i, j] if (isinstance(all_masks, np.ndarray) and all_masks.ndim == 4) else all_masks
                score = yolo_confs[i]
                if all_scores is not None and isinstance(all_scores, np.ndarray) and all_scores.ndim == 2:
                    if i < all_scores.shape[0] and j < all_scores.shape[1]:
                        score = float(all_scores[i, j])

                mask_processed = self._process_mask(mask)
                if score > best_score:
                    best_score = score
                    best_mask = mask_processed

            if best_mask is not None:
                mask_resized = self._resize_mask(best_mask, original_size)
                masks_list.append(mask_resized)
                scores_list.append(best_score)
            else:
                masks_list.append(np.zeros((original_size[1], original_size[0]), dtype=np.uint8))
                scores_list.append(yolo_confs[i])

        return masks_list, scores_list

    # метод обработки изображения
    def _process_single(self, image_path: str) -> Tuple:
        """
        Базовая обработка одного изображения:
        - YOLO-детекция (одна модель)
        - SAM-сегментация (без препроцессинга, без расширения боксов, без NMS)
        """
        local_path = self._get_cached_path(image_path)
        metrics = {'yolo_time': 0.0, 'sam_time': 0.0, 'num_objects': 0}

        # Загрузка изображения
        image_np = np.array(Image.open(local_path).convert('RGB'))
        image = Image.fromarray(image_np)
        original_size = image.size

        # YOLO
        t0 = time.perf_counter()
        results = self.yolo(image_np, conf=self.confidence_threshold, verbose=False)
        boxes = results[0].boxes
        metrics['yolo_time'] = time.perf_counter() - t0

        if boxes is None or len(boxes) == 0:
            return [], [], [], metrics, []

        # Извлечение боксов
        all_bboxes, classes_list, yolo_confs = [], [], []
        for box in boxes:
            cls_id = int(box.cls[0])
            cls_name = self.class_names.get(cls_id, f'class_{cls_id}')
            conf = float(box.conf[0])
            coords = box.xyxy[0].tolist()

            all_bboxes.append(coords)
            classes_list.append(cls_name)
            yolo_confs.append(conf)

        metrics['num_objects'] = len(all_bboxes)

        #SAM
        t0 = time.perf_counter()
        inference_state = self.processor.set_image(image)
        with torch.inference_mode():
            result = self.sam.predict_inst(inference_state, box=all_bboxes)
        masks_list, scores_list = self._process_sam_result(result, len(all_bboxes), original_size, yolo_confs)
        metrics['sam_time'] = time.perf_counter() - t0

        return masks_list, classes_list, scores_list, metrics, all_bboxes

    # Обработка батча с метриками
    def process_batch_with_iou(self, image_paths: List[str], gt_mask_paths: List[Optional[str]]) -> Dict:
        self.preload_images(image_paths)

        metrics = {
            'total_images': len(image_paths), 'successful': 0, 'failed': 0,
            'total_objects': 0, 'total_time': 0.0, 'yolo_time': 0.0, 'sam_time': 0.0,
            'all_ious': [], 'per_class_ious': {cls: [] for cls in TARGET_CLASSES},
            'miou_values': [],
            'class_miou_values': [], 'class_iou_per_class': {cls: [] for cls in TARGET_CLASSES},
            'class_counts': {cls: 0 for cls in TARGET_CLASSES},
            'images_with_iou': 0,
        }

        print(f" Processing {len(image_paths)} images (BASELINE)...")
        print("=" * 60)

        # Прогрев
        _ = self._process_single(image_paths[0])
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        batch_start = time.perf_counter()
        yolo_total, sam_total = 0.0, 0.0

        for idx, (img_path, gt_path) in enumerate(zip(image_paths, gt_mask_paths)):
            try:
                masks, classes, scores, img_metrics, _ = self._process_single(img_path)

                metrics['successful'] += 1
                metrics['total_objects'] += img_metrics['num_objects']
                yolo_total += img_metrics['yolo_time']
                sam_total += img_metrics['sam_time']

                # IoU
                if gt_path and os.path.exists(gt_path) and masks:
                    gt_mask = load_gt_mask(gt_path, Image.open(self._get_cached_path(img_path)).size)

                    # Instance-level
                    iou_metrics = compute_iou_for_pipeline(masks, classes, gt_mask)
                    metrics['all_ious'].extend(iou_metrics['ious'])
                    metrics['miou_values'].append(iou_metrics['miou'])

                    # Class-level
                    class_ious, class_miou = compute_class_level_iou(masks, classes, gt_mask, TARGET_CLASSES)
                    metrics['class_miou_values'].append(class_miou)
                    metrics['images_with_iou'] += 1

                    for cls in TARGET_CLASSES:
                        if cls in class_ious and any(c == cls for c in classes):
                            metrics['class_iou_per_class'][cls].append(class_ious[cls])
                        if iou_metrics['per_class_counts'][cls] > 0:
                            metrics['per_class_ious'][cls].append(iou_metrics['per_class_iou'][cls])

                # Прогресс
                if (idx + 1) % 5 == 0:
                    torch.cuda.empty_cache()
                    gc.collect()
                    elapsed = time.perf_counter() - batch_start
                    fps = (idx + 1) / elapsed
                    recent_miou = np.mean(metrics['class_miou_values'][-10:]) if metrics['class_miou_values'] else 0
                    print(f"  {idx+1:4d}/{len(image_paths)} | {fps:.2f} FPS | mIoU(10): {recent_miou:.4f} | IoU imgs: {metrics['images_with_iou']}")

            except Exception as e:
                print(f"  {Path(img_path).name}: {e}")
                metrics['failed'] += 1

        metrics['total_time'] = time.perf_counter() - batch_start
        metrics['yolo_time'] = yolo_total
        metrics['sam_time'] = sam_total

        if metrics['successful'] > 0:
            metrics['fps'] = metrics['successful'] / metrics['total_time']
            metrics['avg_yolo_ms'] = (yolo_total / metrics['successful']) * 1000
            metrics['avg_sam_ms'] = (sam_total / metrics['successful']) * 1000
            metrics['avg_objects'] = metrics['total_objects'] / metrics['successful']
            metrics['class_miou_avg'] = np.mean(metrics['class_miou_values']) if metrics['class_miou_values'] else 0.0

            metrics['mean_class_iou'] = {}
            for cls in TARGET_CLASSES:
                values = metrics['class_iou_per_class'][cls]
                metrics['mean_class_iou'][cls] = np.mean(values) if values else 0.0
                metrics['class_counts'][cls] = len(values)

            metrics['overall_miou'] = np.mean(metrics['miou_values']) if metrics['miou_values'] else 0.0
            metrics['overall_mean_iou_per_prediction'] = np.mean(metrics['all_ious']) if metrics['all_ious'] else 0.0

            metrics['mean_per_class_iou'] = {
                cls: np.mean(ious) if ious else 0.0
                for cls, ious in metrics['per_class_ious'].items()
            }

        return metrics

    # Вывод реузльтатов и визуализация
    def print_iou_summary(self, metrics: Dict):
        print("\n" + "=" * 60)
        print(" BASELINE BATCH PROCESSING SUMMARY")
        print("=" * 60)
        print(f" Successful: {metrics['successful']}/{metrics['total_images']}")
        print(f" Total objects: {metrics['total_objects']} | Avg/img: {metrics.get('avg_objects', 0):.1f}")
        print(f"  YOLO: {metrics.get('avg_yolo_ms', 0):.1f}ms | SAM: {metrics.get('avg_sam_ms', 0):.1f}ms")
        print(f" FPS: {metrics.get('fps', 0):.2f} | Total time: {metrics['total_time']:.1f}s")
        print(f"\nIoU METRICS ({metrics.get('images_with_iou', 0)} images):")
        print(f"   Class-Level mIoU (primary): {metrics.get('class_miou_avg', 0):.4f}")
        print(f"   ─────────────────────────────")
        print(f"   Instance mIoU (reference):    {metrics.get('overall_miou', 0):.4f}")
        print(f"   Mean IoU/pred (reference):    {metrics.get('overall_mean_iou_per_prediction', 0):.4f}")
        print(f"\n   Per-Class IoU (Class-Level):")
        for cls in TARGET_CLASSES:
            cls_iou = metrics.get('mean_class_iou', {}).get(cls, 0.0)
            count = metrics.get('class_counts', {}).get(cls, 0)
            status = "good" if cls_iou > 0.5 else ("medium" if cls_iou > 0.3 else ("bad" if count > 0 else "—"))
            print(f"     {status} {cls:15s}: {cls_iou:.4f} ({count} imgs)")
        print("=" * 60 + "\n")

    def visualize_with_iou(
        self,
        image_path: str,
        masks_list: List[np.ndarray],
        classes_list: List[str],
        scores_list: Optional[List[float]] = None,
        gt_mask_path: Optional[str] = None,
        iou_metrics: Optional[Dict] = None,
        save_path: Optional[str] = None,
        yolo_boxes: Optional[List[List[float]]] = None
    ):
        """Визуализация с 4 панелями (полностью сохранена)"""
        if not masks_list:
            print("No masks to visualize")
            return

        image = np.array(Image.open(image_path).convert('RGB'))
        h, w = image.shape[:2]
        original_size = (w, h)

        fig, axes = plt.subplots(2, 2, figsize=(20, 16))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

        class_colors = {
            'car': (0, 255, 0), 'person': (255, 0, 0),
            'bicycle': (0, 255, 255), 'motorcycle': (255, 0, 255),
            'bus': (0, 0, 255), 'truck': (255, 165, 0),
            'traffic light': (255, 255, 0),
        }

        # ПАНЕЛЬ 1: Исходное изображение
        ax1.imshow(image)
        ax1.set_title(f'Original Image\n{Path(image_path).name}', fontsize=12, weight='bold')
        ax1.axis('off')

        # ПАНЕЛЬ 2: Боксы YOLO
        ax2.imshow(image)
        if yolo_boxes:
            for i, (box, class_name) in enumerate(zip(yolo_boxes, classes_list)):
                color = class_colors.get(class_name, plt.cm.tab10(i % 10)[:3])
                color_norm = tuple(c / 255 for c in color) if isinstance(color, tuple) else color
                x1, y1, x2, y2 = box
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     fill=False, edgecolor=color_norm, linewidth=2.5)
                ax2.add_patch(rect)
                label = class_name
                if scores_list and i < len(scores_list):
                    label += f" {scores_list[i]:.2f}"
                ax2.text(x1, y1 - 8, label, color='white', fontsize=7, weight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor=color_norm, alpha=0.8))

        from matplotlib.patches import Patch
        yolo_classes = sorted(set(classes_list))
        legend_elements = [Patch(facecolor=tuple(c / 255 for c in class_colors[cls]),
                                 edgecolor=tuple(c / 255 for c in class_colors[cls]),
                                 label=cls) for cls in yolo_classes if cls in class_colors]
        if legend_elements:
            legend = ax2.legend(handles=legend_elements, loc='upper left', fontsize=7,
                              framealpha=0.85, facecolor='black', edgecolor='white', labelcolor='white',
                              title='YOLO detections', title_fontsize=8, ncol=1)
            legend.get_title().set_color('white')
        ax2.set_title(f'YOLO26n Detections\n{len(classes_list)} objects', fontsize=12, weight='bold')
        ax2.axis('off')

        # ПАНЕЛЬ 3: Маски SAM
        ax3.imshow(image)
        sam_classes_present = sorted(set(classes_list))
        for i, (mask, class_name) in enumerate(zip(masks_list, classes_list)):
            color = class_colors.get(class_name, plt.cm.tab10(i % 10)[:3])
            color_normalized = tuple(c / 255 for c in color) if isinstance(color, tuple) else color
            if mask.shape[:2] != (h, w):
                continue
            overlay = np.zeros((h, w, 4))
            overlay[mask > 0] = [*color_normalized, 0.5]
            ax3.imshow(overlay)
            y_idx, x_idx = np.where(mask > 0)
            if len(y_idx) > 0:
                cy, cx = int(np.mean(y_idx)), int(np.mean(x_idx))
                label = class_name
                if iou_metrics and i < len(iou_metrics.get('ious', [])):
                    iou_val = iou_metrics['ious'][i]
                    label += f"\nIoU: {iou_val:.3f}"
                ax3.text(cx, cy, label, color='white', fontsize=7, weight='bold',
                        ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor=color_normalized, alpha=0.7))

        legend_elements = [Patch(facecolor=tuple(c / 255 for c in class_colors[cls]),
                                 edgecolor=tuple(c / 255 for c in class_colors[cls]),
                                 label=cls) for cls in sam_classes_present if cls in class_colors]
        if legend_elements:
            legend = ax3.legend(handles=legend_elements, loc='upper left', fontsize=7,
                              framealpha=0.85, facecolor='black', edgecolor='white', labelcolor='white',
                              title='SAM masks', title_fontsize=8, ncol=1)
            legend.get_title().set_color('white')
        ax3.set_title(f'SAM Segmentation\n{len(masks_list)} masks', fontsize=12, weight='bold')
        ax3.axis('off')

        # ПАНЕЛЬ 4: Ground Truth + IoU
        ax4.imshow(image)
        if gt_mask_path and os.path.exists(gt_mask_path):
            gt_mask = load_gt_mask(gt_mask_path, original_size)
            gt_overlay = np.zeros((h, w, 4))
            gt_classes_present = []
            for train_id, class_name in BDD100K_TRAINID_TO_CLASS.items():
                if class_name in class_colors:
                    color = tuple(c / 255 for c in class_colors[class_name])
                    mask = gt_mask == train_id
                    if mask.any():
                        gt_overlay[mask] = [*color, 0.6]
                        gt_classes_present.append(class_name)
            ax4.imshow(gt_overlay)
            gt_classes_unique = sorted(set(gt_classes_present))
            legend_elements = [Patch(facecolor=tuple(c / 255 for c in class_colors[cls]),
                                     edgecolor=tuple(c / 255 for c in class_colors[cls]),
                                     label=cls) for cls in gt_classes_unique if cls in class_colors]
            if legend_elements:
                legend = ax4.legend(handles=legend_elements, loc='upper left', fontsize=7,
                                  framealpha=0.85, facecolor='black', edgecolor='white', labelcolor='white',
                                  title='GT classes', title_fontsize=8, ncol=1)
                legend.get_title().set_color('white')
            if iou_metrics:
                miou = iou_metrics.get('miou', 0)
                matched = iou_metrics.get('matched_predictions', 0)
                total = iou_metrics.get('total_predictions', 0)
                text = f"Instance mIoU: {miou:.4f}\nMatched: {matched}/{total}\n"
                if gt_mask_path and os.path.exists(gt_mask_path):
                    class_ious, class_miou = compute_class_level_iou(masks_list, classes_list, gt_mask, TARGET_CLASSES)
                    text += f"Class-mIoU: {class_miou:.4f}\n\nClass-Level IoU:\n"
                    for cls in gt_classes_unique:
                        if cls in class_ious:
                            text += f"  {cls}: {class_ious[cls]:.4f}\n"
                ax4.text(0.98, 0.02, text, transform=ax4.transAxes, ha='right', va='bottom',
                        fontsize=8, fontfamily='monospace', color='white',
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='black', alpha=0.85))
            ax4.set_title(f'Ground Truth + IoU\n{len(gt_classes_unique)} classes', fontsize=12, weight='bold')
        else:
            ax4.text(0.5, 0.5, 'GT Mask\nNot Available', transform=ax4.transAxes,
                    ha='center', va='center', fontsize=14, color='gray', weight='bold')
            ax4.set_title('Ground Truth', fontsize=12, weight='bold')
        ax4.axis('off')

        fig.suptitle('YOLO26n + EfficientSAM3 — BASELINE (No Improvements)',
                    fontsize=14, weight='bold', y=0.98)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
            print(f" Visualization saved to: {save_path}")
        else:
            plt.show()
        plt.close()

    def cleanup(self):
        if os.path.exists(self.cache_dir):
            shutil.rmtree(self.cache_dir)

## 2.3 Основной класс с улучшениями

In [46]:
class YOLO_SAM_Pipeline:
    """YOLO26n + EfficientSAM3 — детекция, сегментация, IoU метрики"""

    def __init__(self,
             sam_checkpoint='/content/checkpoints/efficient_sam3_efficientvit_s.pt',
             confidence_threshold=0.5,
             backbone_type='efficientvit',
             model_name='b0',
             cache_dir: str = '/content/image_cache'):

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"  Device: {self.device}")

        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

        # YOLO — основная модель (nano для скорости)
        print(" Loading YOLO26n (main)...")
        self.yolo = YOLO('yolo26n.pt').to(self.device)
        self.yolo.model.half()

        # YOLO — ансамбль для мелких объектов (medium для точности)
        print(" Loading YOLO26m (small objects)...")
        self.yolo_small = YOLO('yolo26m.pt').to(self.device)
        self.yolo_small.model.half()

        self.confidence_threshold = confidence_threshold

        # SAM
        print(" Loading EfficientSAM3...")
        self.sam = build_efficientsam3_image_model(
            checkpoint_path=sam_checkpoint,
            backbone_type=backbone_type,
            model_name=model_name,
            enable_inst_interactivity=True,
        ).to(self.device).eval()

        self.processor = Sam3Processor(self.sam)

        self.class_names = {
            0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle',
            5: 'bus', 7: 'truck', 9: 'traffic light'
        }

        # Параметры для ансамбля
        self.small_object_threshold = 80
        self.use_small_object_detector = True

        print(f" Pipeline ready (conf={self.confidence_threshold})")

    def _get_cached_path(self, remote_path: str) -> str:
        filename = Path(remote_path).name
        local_path = os.path.join(self.cache_dir, filename)
        if not os.path.exists(local_path):
            shutil.copy2(remote_path, local_path)
        return local_path

    def preload_images(self, image_paths: List[str]) -> None:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=4) as executor:
            list(executor.map(self._get_cached_path, image_paths))

    def _box_iou(self, a, b):
        """IoU двух боксов"""
        xa = max(a[0], b[0]); ya = max(a[1], b[1])
        xb = min(a[2], b[2]); yb = min(a[3], b[3])
        inter = max(0, xb - xa) * max(0, yb - ya)
        area_a = (a[2]-a[0]) * (a[3]-a[1])
        area_b = (b[2]-b[0]) * (b[3]-b[1])
        return inter / (area_a + area_b - inter) if (area_a + area_b - inter) > 0 else 0

    def _fuse_boxes_with_small(self, boxes_main, boxes_small, iou_thr=0.5) -> List[Dict]:
        """
        Объединение боксов от основной модели (YOLO26n) и детектора мелких объектов (YOLO26m).
        YOLO26m используется только для мелких объектов, не дублируя крупные.
        """
        all_boxes = []

        # Основная модель (вес 2.0) — все объекты
        if boxes_main is not None and len(boxes_main) > 0:
            for box in boxes_main:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w, h = x2 - x1, y2 - y1
                all_boxes.append({
                    'coords': box.xyxy[0].tolist(),
                    'conf': float(box.conf[0]),
                    'cls': int(box.cls[0]),
                    'weight': 2.0,
                    'size': max(w, h)  # Сохраняем размер для фильтрации
                })

        # Детектор мелких объектов (вес 1.5) — только мелкие объекты
        if boxes_small is not None and len(boxes_small) > 0:
            for box in boxes_small:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w, h = x2 - x1, y2 - y1

                # Добавляем только мелкие объекты
                if max(w, h) < self.small_object_threshold:
                    all_boxes.append({
                        'coords': box.xyxy[0].tolist(),
                        'conf': float(box.conf[0]) * 1.2,  # Чуть повышаем уверенность
                        'cls': int(box.cls[0]),
                        'weight': 1.5,
                        'size': max(w, h)
                    })
                else:
                    # Для крупных тоже добавляем, но с меньшим весом (для устранения пропусков)
                    all_boxes.append({
                        'coords': box.xyxy[0].tolist(),
                        'conf': float(box.conf[0]),
                        'cls': int(box.cls[0]),
                        'weight': 0.8,
                        'size': max(w, h)
                    })

        if not all_boxes:
            return []

        # Кластеризация по IoU и усреднение
        fused = []
        used = set()

        for i, box_a in enumerate(all_boxes):
            if i in used:
                continue
            cluster = [box_a]
            used.add(i)

            for j, box_b in enumerate(all_boxes):
                if j in used or box_a['cls'] != box_b['cls']:
                    continue
                iou = self._box_iou(box_a['coords'], box_b['coords'])
                if iou > iou_thr:
                    cluster.append(box_b)
                    used.add(j)

            # Усредняем кластер
            total_weight = sum(b['weight'] for b in cluster)
            avg_conf = sum(b['conf'] * b['weight'] for b in cluster) / total_weight
            avg_coords = [
                sum(b['coords'][k] * b['weight'] for b in cluster) / total_weight
                for k in range(4)
            ]

            fused.append({
                'coords': avg_coords,
                'conf': min(avg_conf, 1.0),
                'cls': cluster[0]['cls']
            })

        return fused

    def _mask_nms_per_class(self, masks, classes, scores, iou_threshold=0.3):
        """Удаляет дубликаты масок внутри каждого класса"""
        if len(masks) == 0:
            return []

        class_groups = {}
        for i, (mask, cls, score) in enumerate(zip(masks, classes, scores)):
            class_groups.setdefault(cls, {'indices': [], 'masks': [], 'scores': []})
            class_groups[cls]['indices'].append(i)
            class_groups[cls]['masks'].append(mask)
            class_groups[cls]['scores'].append(score)

        keep_global = []
        for cls, group in class_groups.items():
            indices, cls_masks, cls_scores = group['indices'], group['masks'], group['scores']
            n = len(cls_masks)
            if n <= 1:
                keep_global.extend(indices)
                continue

            order = np.argsort(cls_scores)[::-1]
            keep = []
            while len(order) > 0:
                current_idx = order[0]
                keep.append(indices[current_idx])
                if len(order) == 1:
                    break

                current_mask = cls_masks[current_idx]
                current_area = current_mask.sum()
                ious = []
                for other_idx in order[1:]:
                    other_mask = cls_masks[other_idx]
                    intersection = np.logical_and(current_mask, other_mask).sum()
                    union = current_area + other_mask.sum() - intersection
                    ious.append(intersection / union if union > 0 else 0)

                remaining = np.where(np.array(ious) <= iou_threshold)[0]
                order = order[remaining + 1]
            keep_global.extend(keep)

        return sorted(keep_global)

    def _process_single(self, image_path: str, preprocess: bool = False) -> Tuple:
        local_path = self._get_cached_path(image_path)
        metrics = {'yolo_time': 0.0, 'sam_time': 0.0, 'num_objects': 0}

        # Загрузка изображения
        image_np = np.array(Image.open(local_path).convert('RGB'))

        # Базовые пороги по классам
        class_conf = {
            'car': 0.35,
            'truck': 0.35,
            'bus': 0.35,
            'person': 0.20,
            'traffic light': 0.25,
            'bicycle': 0.20,
            'motorcycle': 0.20,
        }

        # Адаптивный confidence threshold
        current_conf = self.confidence_threshold

        # Предобработка (опционально)
        condition = None
        if preprocess:
            condition = detect_visibility_condition(image_np)
            image_np = preprocess_for_visibility(image_np, condition)
            metrics['visibility'] = condition.value

            if condition == VisibilityCondition.NIGHT:
                current_conf = 0.25
                for cls in class_conf:
                    class_conf[cls] = max(0.15, class_conf[cls] - 0.05)
            elif condition == VisibilityCondition.FOG:
                current_conf = 0.30
                for cls in class_conf:
                    class_conf[cls] = max(0.15, class_conf[cls] - 0.03)
            elif condition in [VisibilityCondition.RAIN, VisibilityCondition.SNOW]:
                current_conf = 0.35

        # Конвертация в PIL
        image = Image.fromarray(image_np)
        original_size = image.size

        # === АНСАМБЛЬ YOLO (nano + medium) ===
        t0 = time.perf_counter()

        # Основная модель (nano) — быстро, всё изображение
        results_main = self.yolo(image_np, conf=current_conf, verbose=False)
        boxes_main = results_main[0].boxes

        # Детектор мелких объектов (medium) — на полном разрешении
        if self.use_small_object_detector and hasattr(self, 'yolo_small'):
            results_small = self.yolo_small(
                image_np,
                conf=current_conf * 0.6,  # Ниже порог для поиска мелких
                iou=0.4,                   # Мягче NMS
                verbose=False
            )
            boxes_small = results_small[0].boxes

            # Слияние с учётом мелких объектов
            fused_boxes = self._fuse_boxes_with_small(boxes_main, boxes_small, iou_thr=0.5)
            boxes = fused_boxes
        else:
            boxes = boxes_main

        metrics['yolo_time'] = time.perf_counter() - t0

        if boxes is None or len(boxes) == 0:
            return [], [], [], metrics, []

        # Преобразование боксов
        all_boxes = []
        for box in boxes:
            if isinstance(box, dict):
                cls_id = box['cls']
                conf = box['conf']
                coords = box['coords']
            else:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                coords = box.xyxy[0].tolist()

            cls_name = self.class_names.get(cls_id, f'class_{cls_id}')
            min_conf = class_conf.get(cls_name, current_conf)

            if conf >= min_conf:
                all_boxes.append((coords, cls_name, conf, cls_id))

        if not all_boxes:
            return [], [], [], metrics, []

        metrics['num_objects'] = len(all_boxes)

        all_bboxes, classes_list, yolo_confs = [], [], []
        for coords, cls_name, conf, _ in all_boxes:
            x1, y1, x2, y2 = coords

            # Расширяем бокс для мелких классов
            if cls_name in ['traffic light', 'person', 'bicycle', 'motorcycle']:
                w, h = x2 - x1, y2 - y1
                x1 = max(0, x1 - w * 0.15)
                y1 = max(0, y1 - h * 0.15)
                x2 = x2 + w * 0.15
                y2 = y2 + h * 0.15

            all_bboxes.append([x1, y1, x2, y2])
            classes_list.append(cls_name)
            yolo_confs.append(conf)

        # SAM — передаём PIL Image
        inference_state = self.processor.set_image(image)
        t0 = time.perf_counter()
        with torch.inference_mode():
            result = self.sam.predict_inst(inference_state, box=all_bboxes)
        masks_list, scores_list = self._process_sam_result(result, len(all_bboxes), original_size, yolo_confs)
        metrics['sam_time'] = time.perf_counter() - t0

        # Mask NMS
        if len(masks_list) > 1:
            from collections import defaultdict
            class_masks = defaultdict(list)
            for i, (mask, cls, score) in enumerate(zip(masks_list, classes_list, scores_list)):
                class_masks[cls].append((i, mask, score))

            keep_indices = []
            for cls, items in class_masks.items():
                if len(items) <= 1:
                    keep_indices.extend([item[0] for item in items])
                    continue

                areas = [item[1].sum() for item in items]
                best_idx = np.argmax(areas)
                keep_indices.append(items[best_idx][0])

                best_mask = items[best_idx][1]
                for i, (idx, mask, score) in enumerate(items):
                    if i == best_idx:
                        continue
                    iou_with_best = compute_iou(mask, best_mask)
                    if iou_with_best < 0.5:
                        keep_indices.append(idx)

            masks_list = [masks_list[i] for i in sorted(keep_indices)]
            classes_list = [classes_list[i] for i in sorted(keep_indices)]
            scores_list = [scores_list[i] for i in sorted(keep_indices)]

        return masks_list, classes_list, scores_list, metrics, all_bboxes

    def _fuse_boxes(self, boxes_main, boxes_aux, iou_thr=0.5) -> List[Dict]:
        """Объединение боксов от основной и вспомогательной моделей."""
        all_boxes = []

        # Основная модель (вес 2.0)
        if boxes_main is not None and len(boxes_main) > 0:
            for box in boxes_main:
                all_boxes.append({
                    'coords': box.xyxy[0].tolist(),
                    'conf': float(box.conf[0]),
                    'cls': int(box.cls[0]),
                    'weight': 2.0
                })

        # Вспомогательная модель (вес 0.5, боксы в 2 раза меньше → умножаем на 2)
        if boxes_aux is not None and len(boxes_aux) > 0:
            for box in boxes_aux:
                coords = box.xyxy[0].tolist()
                # Масштабируем обратно (было 640×360 → 1280×720)
                coords = [c * 2 for c in coords]
                all_boxes.append({
                    'coords': coords,
                    'conf': float(box.conf[0]),
                    'cls': int(box.cls[0]),
                    'weight': 0.5
                })

        if not all_boxes:
            return []

        # Кластеризация по IoU и усреднение
        fused = []
        used = set()

        for i, box_a in enumerate(all_boxes):
            if i in used:
                continue
            cluster = [box_a]
            used.add(i)

            for j, box_b in enumerate(all_boxes):
                if j in used or box_a['cls'] != box_b['cls']:
                    continue
                iou = self._box_iou(box_a['coords'], box_b['coords'])
                if iou > iou_thr:
                    cluster.append(box_b)
                    used.add(j)

            # Усредняем кластер
            total_weight = sum(b['weight'] for b in cluster)
            avg_conf = sum(b['conf'] * b['weight'] for b in cluster) / total_weight
            avg_coords = [
                sum(b['coords'][k] * b['weight'] for b in cluster) / total_weight
                for k in range(4)
            ]

            fused.append({
                'coords': avg_coords,
                'conf': min(avg_conf, 1.0),
                'cls': cluster[0]['cls']
            })

        return fused

    def _box_iou(self, a: List[float], b: List[float]) -> float:
        """IoU двух боксов."""
        xa = max(a[0], b[0]); ya = max(a[1], b[1])
        xb = min(a[2], b[2]); yb = min(a[3], b[3])
        inter = max(0, xb - xa) * max(0, yb - ya)
        area_a = (a[2] - a[0]) * (a[3] - a[1])
        area_b = (b[2] - b[0]) * (b[3] - b[1])
        union = area_a + area_b - inter
        return inter / union if union > 0 else 0.0

    def _process_sam_result(self, result, num_objects, original_size, yolo_confs):
        masks_list, scores_list = [], []

        if isinstance(result, (tuple, list)) and len(result) >= 2:
            all_masks, all_scores = result[0], result[1]
        else:
            all_masks, all_scores = result, None

        if all_masks is None:
            return [], []

        if isinstance(all_masks, torch.Tensor):
            all_masks = all_masks.cpu().numpy()
        if isinstance(all_scores, torch.Tensor):
            all_scores = all_scores.cpu().numpy()

        num_masks_per_obj = all_masks.shape[1] if isinstance(all_masks, np.ndarray) and all_masks.ndim == 4 else 1

        for i in range(num_objects):
            best_mask, best_score = None, 0.0
            for j in range(num_masks_per_obj):
                mask = all_masks[i, j] if (isinstance(all_masks, np.ndarray) and all_masks.ndim == 4) else all_masks
                score = yolo_confs[i]
                if all_scores is not None and isinstance(all_scores, np.ndarray) and all_scores.ndim == 2:
                    if i < all_scores.shape[0] and j < all_scores.shape[1]:
                        score = float(all_scores[i, j])

                mask_processed = self._process_mask(mask)
                if score > best_score:
                    best_score = score
                    best_mask = mask_processed

            if best_mask is not None:
                mask_resized = self._resize_mask(best_mask, original_size)
                masks_list.append(mask_resized)
                scores_list.append(best_score)
            else:
                masks_list.append(np.zeros((original_size[1], original_size[0]), dtype=np.uint8))
                scores_list.append(yolo_confs[i])

        return masks_list, scores_list

    def _process_mask(self, mask):
        mask = np.squeeze(mask)
        if mask.ndim == 3:
            mask = mask[0] if mask.shape[0] == 3 else mask[:, :, 0]
        if mask.dtype == bool:
            return mask.astype(np.uint8)
        elif mask.dtype in [np.float32, np.float64, float]:
            return (mask > 0.5).astype(np.uint8)
        else:
            return (mask > 127).astype(np.uint8)

    def _resize_mask(self, mask, target_size):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
        mask_resized = mask_pil.resize(target_size, Image.NEAREST)
        return (np.array(mask_resized) > 127).astype(np.uint8)

    def process_batch_with_iou(self, image_paths: List[str], gt_mask_paths: List[Optional[str]], preprocess: bool = False) -> Dict:
        self.preload_images(image_paths)

        metrics = {
            'total_images': len(image_paths), 'successful': 0, 'failed': 0,
            'total_objects': 0, 'total_time': 0.0, 'yolo_time': 0.0, 'sam_time': 0.0,
            # Instance-level метрики (справочные)
            'all_ious': [], 'per_class_ious': {cls: [] for cls in TARGET_CLASSES},
            'miou_values': [],
            # Class-level метрики (основные)
            'class_miou_values': [], 'class_iou_per_class': {cls: [] for cls in TARGET_CLASSES},
            'class_counts': {cls: 0 for cls in TARGET_CLASSES},
            'images_with_iou': 0,
            'visibility_stats': {c.value: {'count': 0, 'miou': []} for c in VisibilityCondition},
        }

        print(f" Processing {len(image_paths)} images...")
        if preprocess:
            print("    Adaptive preprocessing ENABLED")

        # Прогрев
        _ = self._process_single(image_paths[0], preprocess=preprocess)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        batch_start = time.perf_counter()
        yolo_total, sam_total = 0.0, 0.0

        for idx, (img_path, gt_path) in enumerate(zip(image_paths, gt_mask_paths)):
            try:
                masks, classes, scores, img_metrics, _ = self._process_single(img_path, preprocess=preprocess)

                metrics['successful'] += 1
                metrics['total_objects'] += img_metrics['num_objects']
                yolo_total += img_metrics['yolo_time']
                sam_total += img_metrics['sam_time']

                # IoU
                if gt_path and os.path.exists(gt_path) and masks:
                    gt_mask = load_gt_mask(gt_path, Image.open(self._get_cached_path(img_path)).size)

                    # Instance-level (справочно)
                    iou_metrics = compute_iou_for_pipeline(masks, classes, gt_mask)
                    metrics['all_ious'].extend(iou_metrics['ious'])
                    metrics['miou_values'].append(iou_metrics['miou'])

                    # Class-level (основная метрика)
                    class_ious, class_miou = compute_class_level_iou(masks, classes, gt_mask, TARGET_CLASSES)
                    metrics['class_miou_values'].append(class_miou)
                    metrics['images_with_iou'] += 1

                    # Per-class агрегация
                    for cls in TARGET_CLASSES:
                        # Class-level IoU для класса
                        if cls in class_ious and any(c == cls for c in classes):
                            metrics['class_iou_per_class'][cls].append(class_ious[cls])

                        # Instance-level IoU для класса (справочно)
                        if iou_metrics['per_class_counts'][cls] > 0:
                            metrics['per_class_ious'][cls].append(iou_metrics['per_class_iou'][cls])

                    # Статистика по условиям видимости
                    if img_metrics.get('visibility'):
                        vis = img_metrics['visibility']
                        metrics['visibility_stats'][vis]['count'] += 1
                        metrics['visibility_stats'][vis]['miou'].append(class_miou)

                # Прогресс каждые 5 изображений
                if (idx + 1) % 5 == 0:
                    torch.cuda.empty_cache()
                    gc.collect()
                    elapsed = time.perf_counter() - batch_start
                    fps = (idx + 1) / elapsed
                    recent_miou = np.mean(metrics['class_miou_values'][-10:]) if metrics['class_miou_values'] else 0
                    print(f"   {idx+1:4d}/{len(image_paths)} | {fps:.2f} FPS | mIoU(10): {recent_miou:.4f} | IoU imgs: {metrics['images_with_iou']}")

            except Exception as e:
                print(f"   {Path(img_path).name}: {e}")
                metrics['failed'] += 1

        metrics['total_time'] = time.perf_counter() - batch_start
        metrics['yolo_time'] = yolo_total
        metrics['sam_time'] = sam_total

        if metrics['successful'] > 0:
            metrics['fps'] = metrics['successful'] / metrics['total_time']
            metrics['avg_yolo_ms'] = (yolo_total / metrics['successful']) * 1000
            metrics['avg_sam_ms'] = (sam_total / metrics['successful']) * 1000
            metrics['avg_objects'] = metrics['total_objects'] / metrics['successful']

            # Основная метрика — Class-Level mIoU
            metrics['class_miou_avg'] = np.mean(metrics['class_miou_values']) if metrics['class_miou_values'] else 0.0

            # Агрегация Class-Level IoU по классам
            metrics['mean_class_iou'] = {}
            for cls in TARGET_CLASSES:
                values = metrics['class_iou_per_class'][cls]
                metrics['mean_class_iou'][cls] = np.mean(values) if values else 0.0
                metrics['class_counts'][cls] = len(values)

            # Instance-Level метрики (справочные)
            metrics['overall_miou'] = np.mean(metrics['miou_values']) if metrics['miou_values'] else 0.0
            metrics['overall_mean_iou_per_prediction'] = np.mean(metrics['all_ious']) if metrics['all_ious'] else 0.0

            # Per-class Instance IoU (справочно)
            metrics['mean_per_class_iou'] = {
                cls: np.mean(ious) if ious else 0.0
                for cls, ious in metrics['per_class_ious'].items()
            }

        return metrics

    def print_iou_summary(self, metrics: Dict):
        print("\n" + "=" * 60)
        print(" BATCH PROCESSING SUMMARY WITH IoU")
        print("=" * 60)
        print(f" Successful: {metrics['successful']}/{metrics['total_images']}")
        print(f" Total objects: {metrics['total_objects']} | Avg/img: {metrics.get('avg_objects', 0):.1f}")
        print(f"  YOLO: {metrics.get('avg_yolo_ms', 0):.1f}ms | SAM: {metrics.get('avg_sam_ms', 0):.1f}ms")
        print(f" FPS: {metrics.get('fps', 0):.2f} | Total time: {metrics['total_time']:.1f}s")
        print(f"\n IoU METRICS ({metrics.get('images_with_iou', 0)} images):")
        print(f"   Class-Level mIoU (primary): {metrics.get('class_miou_avg', 0):.4f}")
        print(f"   ─────────────────────────────")
        print(f"   Instance mIoU (reference):    {metrics.get('overall_miou', 0):.4f}")
        print(f"   Mean IoU/pred (reference):    {metrics.get('overall_mean_iou_per_prediction', 0):.4f}")
        print(f"\n   Per-Class IoU (Class-Level):")
        for cls in TARGET_CLASSES:
            cls_iou = metrics.get('mean_class_iou', {}).get(cls, 0.0)
            count = metrics.get('class_counts', {}).get(cls, 0)
            status = "good" if cls_iou > 0.5 else ("medium" if cls_iou > 0.3 else ("bad" if count > 0 else "—"))
            print(f"     {status} {cls:15s}: {cls_iou:.4f} ({count} imgs)")

        # Статистика по условиям видимости
        visibility_stats = metrics.get('visibility_stats', {})
        if any(v['count'] > 0 for v in visibility_stats.values()):
            print(f"\n🌦️  Visibility Breakdown:")
            for condition in VisibilityCondition:
                stats = visibility_stats.get(condition.value, {})
                count = stats.get('count', 0)
                miou_vals = stats.get('miou', [])
                avg_miou = np.mean(miou_vals) if miou_vals else 0.0
                if count > 0:
                    print(f"   {condition.value:8s}: {avg_miou:.4f} ({count} imgs)")
        print("=" * 60 + "\n")

    def visualize_with_iou(
        self,
        image_path: str,
        masks_list: List[np.ndarray],
        classes_list: List[str],
        scores_list: Optional[List[float]] = None,
        gt_mask_path: Optional[str] = None,
        iou_metrics: Optional[Dict] = None,
        save_path: Optional[str] = None,
        yolo_boxes: Optional[List[List[float]]] = None
    ):
        """
        Расширенная визуализация с 4 панелями:
        1. Original Image
        2. YOLO Detections (боксы)
        3. SAM Segmentation (предсказанные маски)
        4. Ground Truth (размеченная маска)
        + IoU метрики на панели 4

        Args:
            image_path: путь к исходному изображению
            masks_list: список предсказанных масок
            classes_list: список классов
            scores_list: confidence scores
            gt_mask_path: путь к GT маске
            iou_metrics: метрики IoU из compute_iou_for_pipeline
            save_path: путь для сохранения (если None — показывает на экран)
            yolo_boxes: список боксов [x1, y1, x2, y2] от YOLO
        """
        if not masks_list:
            print("No masks to visualize")
            return

        # Загрузка изображения
        image = np.array(Image.open(image_path).convert('RGB'))
        h, w = image.shape[:2]
        original_size = (w, h)

        # Создаём фигуру с 4 панелями (2x2)
        fig, axes = plt.subplots(2, 2, figsize=(20, 16))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

        # Цвета для классов
        class_colors = {
            'car': (0, 255, 0),           # зелёный
            'person': (255, 0, 0),        # красный
            'bicycle': (0, 255, 255),     # жёлтый
            'motorcycle': (255, 0, 255),  # пурпурный
            'bus': (0, 0, 255),           # синий
            'truck': (255, 165, 0),       # оранжевый
            'traffic light': (255, 255, 0), # ярко-жёлтый
        }

        # ========================================
        # ПАНЕЛЬ 1: Исходное изображение
        # ========================================
        ax1.imshow(image)
        ax1.set_title(f'Original Image\n{Path(image_path).name}', fontsize=12, weight='bold')
        ax1.axis('off')

        # ========================================
        # ПАНЕЛЬ 2: Боксы YOLO
        # ========================================
        ax2.imshow(image)

        if yolo_boxes:
            for i, (box, class_name) in enumerate(zip(yolo_boxes, classes_list)):
                color = class_colors.get(class_name, plt.cm.tab10(i % 10)[:3])
                color_norm = tuple(c / 255 for c in color) if isinstance(color, tuple) else color

                x1, y1, x2, y2 = box
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     fill=False, edgecolor=color_norm, linewidth=2.5)
                ax2.add_patch(rect)

                # Подпись класса и confidence
                label = class_name
                if scores_list and i < len(scores_list):
                    label += f" {scores_list[i]:.2f}"
                ax2.text(x1, y1 - 8, label, color='white', fontsize=7, weight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor=color_norm, alpha=0.8))

        # Легенда YOLO
        from matplotlib.patches import Patch
        yolo_classes = sorted(set(classes_list))
        legend_elements = []
        for cls in yolo_classes:
            if cls in class_colors:
                color = tuple(c / 255 for c in class_colors[cls])
                legend_elements.append(Patch(facecolor=color, edgecolor=color, label=cls))

        if legend_elements:
            legend = ax2.legend(
                handles=legend_elements,
                loc='upper left',
                fontsize=7,
                framealpha=0.85,
                facecolor='black',
                edgecolor='white',
                labelcolor='white',
                title='YOLO detections',
                title_fontsize=8,
                ncol=1
            )
            legend.get_title().set_color('white')

        ax2.set_title(f'YOLO26n Detections\n{len(classes_list)} objects', fontsize=12, weight='bold')
        ax2.axis('off')

        # ========================================
        # ПАНЕЛЬ 3: Предсказанные маски SAM
        # ========================================
        ax3.imshow(image)

        sam_classes_present = sorted(set(classes_list))

        for i, (mask, class_name) in enumerate(zip(masks_list, classes_list)):
            color = class_colors.get(class_name, plt.cm.tab10(i % 10)[:3])
            color_normalized = tuple(c / 255 for c in color) if isinstance(color, tuple) else color

            if mask.shape[:2] != (h, w):
                continue

            # Создаём цветной оверлей
            overlay = np.zeros((h, w, 4))
            overlay[mask > 0] = [*color_normalized, 0.5]
            ax3.imshow(overlay)

            # Подпись с IoU
            y_idx, x_idx = np.where(mask > 0)
            if len(y_idx) > 0:
                cy, cx = int(np.mean(y_idx)), int(np.mean(x_idx))
                label = class_name
                if iou_metrics and i < len(iou_metrics.get('ious', [])):
                    iou_val = iou_metrics['ious'][i]
                    label += f"\nIoU: {iou_val:.3f}"
                ax3.text(cx, cy, label, color='white', fontsize=7, weight='bold',
                        ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor=color_normalized, alpha=0.7))

        # Легенда SAM
        legend_elements = []
        for cls in sam_classes_present:
            if cls in class_colors:
                color = tuple(c / 255 for c in class_colors[cls])
                legend_elements.append(Patch(facecolor=color, edgecolor=color, label=cls))

        if legend_elements:
            legend = ax3.legend(
                handles=legend_elements,
                loc='upper left',
                fontsize=7,
                framealpha=0.85,
                facecolor='black',
                edgecolor='white',
                labelcolor='white',
                title='SAM masks',
                title_fontsize=8,
                ncol=1
            )
            legend.get_title().set_color('white')

        total_preds = len(masks_list)
        ax3.set_title(f'SAM Segmentation\n{total_preds} masks', fontsize=12, weight='bold')
        ax3.axis('off')

        # ========================================
        # ПАНЕЛЬ 4: Ground Truth + IoU метрики
        # ========================================
        ax4.imshow(image)

        if gt_mask_path and os.path.exists(gt_mask_path):
            gt_mask = load_gt_mask(gt_mask_path, original_size)
            gt_overlay = np.zeros((h, w, 4))
            gt_classes_present = []

            for train_id, class_name in BDD100K_TRAINID_TO_CLASS.items():
                if class_name in class_colors:
                    color = tuple(c / 255 for c in class_colors[class_name])
                    mask = gt_mask == train_id
                    if mask.any():
                        gt_overlay[mask] = [*color, 0.6]
                        gt_classes_present.append(class_name)

            ax4.imshow(gt_overlay)

            # Легенда GT
            gt_classes_unique = sorted(set(gt_classes_present))
            legend_elements = []
            for cls in gt_classes_unique:
                if cls in class_colors:
                    color = tuple(c / 255 for c in class_colors[cls])
                    legend_elements.append(Patch(facecolor=color, edgecolor=color, label=cls))

            if legend_elements:
                legend = ax4.legend(
                    handles=legend_elements,
                    loc='upper left',
                    fontsize=7,
                    framealpha=0.85,
                    facecolor='black',
                    edgecolor='white',
                    labelcolor='white',
                    title='GT classes',
                    title_fontsize=8,
                    ncol=1
                )
                legend.get_title().set_color('white')

            # IoU метрики на GT-панели (справа внизу)
            if iou_metrics:
                miou = iou_metrics.get('miou', 0)
                matched = iou_metrics.get('matched_predictions', 0)
                total = iou_metrics.get('total_predictions', 0)

                text = f"Instance mIoU: {miou:.4f}\n"
                text += f"Matched: {matched}/{total}\n"

                # Добавляем Class-Level IoU для присутствующих классов
                if gt_mask_path and os.path.exists(gt_mask_path):
                    class_ious, class_miou = compute_class_level_iou(
                        masks_list, classes_list, gt_mask, TARGET_CLASSES
                    )
                    text += f"Class-mIoU: {class_miou:.4f}\n\n"
                    text += "Class-Level IoU:\n"
                    for cls in gt_classes_unique:
                        if cls in class_ious:
                            text += f"  {cls}: {class_ious[cls]:.4f}\n"

                ax4.text(0.98, 0.02, text, transform=ax4.transAxes,
                        ha='right', va='bottom', fontsize=8,
                        fontfamily='monospace', color='white',
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='black', alpha=0.85))

            ax4.set_title(f'Ground Truth + IoU\n{len(gt_classes_unique)} classes', fontsize=12, weight='bold')
        else:
            ax4.text(0.5, 0.5, 'GT Mask\nNot Available',
                    transform=ax4.transAxes, ha='center', va='center',
                    fontsize=14, color='gray', weight='bold')
            ax4.set_title('Ground Truth', fontsize=12, weight='bold')

        ax4.axis('off')

        # Общий заголовок
        fig.suptitle('YOLO26n + EfficientSAM3 — Segmentation Analysis',
                    fontsize=14, weight='bold', y=0.98)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
            print(f" Visualization saved to: {save_path}")
        else:
            plt.show()

        plt.close()

    def cleanup(self):
        if os.path.exists(self.cache_dir):
            shutil.rmtree(self.cache_dir)

# 3. Запуск пайплайна

## 3.1 Настройка конфигурации

In [30]:
BDD100K_COLOR_TO_TRAINID = {
    (220, 20, 60): 11,    # person
    (255, 0, 0): 12,      # rider
    (0, 0, 142): 13,      # car
    (119, 11, 32): 18,    # bicycle
    (0, 0, 230): 17,      # motorcycle
    (0, 60, 100): 14,     # bus
    (0, 0, 70): 15,       # truck
    (250, 170, 30): 6,    # traffic light
}

BDD100K_TRAINID_TO_CLASS = {
    11: 'person', 12: 'person',
    13: 'car', 18: 'bicycle', 17: 'motorcycle',
    14: 'bus', 15: 'truck', 6: 'traffic light',
}

TARGET_CLASSES = ['person', 'car', 'bicycle', 'motorcycle', 'bus', 'truck', 'traffic light']

CLASS_SPECIFIC_MIN_AREA = {
    'person': 20, 'bicycle': 50, 'motorcycle': 50,
    'car': 100, 'bus': 200, 'truck': 150, 'traffic light': 3,
}


## 3.2 Запуск базового пайплайна на изображениях, сделанных днем в хорошую вижимость

In [36]:
if __name__ == "__main__":
    print("\n=== BASELINE: YOLO26n + EfficientSAM3 (no improvements) ===\n")
    clear_gpu_memory()

    pipeline = YOLO_SAM_Pipeline_Baseline(
        sam_checkpoint='/content/checkpoints/efficient_sam3_efficientvit_s.pt',
        backbone_type='efficientvit',
        model_name='b0',
        confidence_threshold=0.5
    )

    image_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/images/val'
    mask_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/labels/val'

    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))[:1000]
    print(f"Found {len(image_paths)} images")

    gt_mask_paths = []
    for img_path in image_paths:
        img_name = Path(img_path).stem
        mask_found = False
        for suffix in ['_train_color.png', '_train_id.png', '.png']:
            mask_path = os.path.join(mask_dir, f"{img_name}{suffix}")
            if os.path.exists(mask_path):
                gt_mask_paths.append(mask_path)
                mask_found = True
                break
        if not mask_found:
            matching = glob.glob(os.path.join(mask_dir, f"{img_name}*.png"))
            gt_mask_paths.append(matching[0] if matching else None)

    print(f"   Masks found: {sum(1 for p in gt_mask_paths if p)}/{len(image_paths)}")

    metrics = pipeline.process_batch_with_iou(image_paths, gt_mask_paths)
    pipeline.print_iou_summary(metrics)

    # Визуализация
    viz_dir = '/content/viz_results_baseline'
    os.makedirs(viz_dir, exist_ok=True)
    num_viz = min(5, len(image_paths))
    print(f"\n Saving {num_viz} visualizations to {viz_dir}...")

    for i in range(num_viz):
        img_path = image_paths[i]
        gt_path = gt_mask_paths[i]
        masks, classes, scores, img_metrics, yolo_boxes = pipeline._process_single(img_path)
        if masks:
            iou_metrics = None
            if gt_path and os.path.exists(gt_path):
                gt_mask = load_gt_mask(gt_path, Image.open(img_path).size)
                iou_metrics = compute_iou_for_pipeline(masks, classes, gt_mask)
            save_path = os.path.join(viz_dir, f"baseline_{i+1:03d}_{Path(img_path).stem}.png")
            pipeline.visualize_with_iou(img_path, masks, classes, scores, gt_path, iou_metrics, save_path, yolo_boxes)

    print(f" Visualizations saved to: {viz_dir}")
    pipeline.cleanup()
    clear_gpu_memory()


=== BASELINE: YOLO26n + EfficientSAM3 (no improvements) ===

🖥️  Device: cuda
🔄 Loading YOLO26n...
🔄 Loading EfficientSAM3...
✅ Baseline Pipeline ready (conf=0.5)

Found 1000 images
  ✅ Masks found: 1000/1000
🚀 Processing 1000 images (BASELINE)...
  📊    5/1000 | 1.54 FPS | mIoU(10): 0.8435 | IoU imgs: 5
  📊   10/1000 | 1.62 FPS | mIoU(10): 0.8194 | IoU imgs: 9
  📊   15/1000 | 1.80 FPS | mIoU(10): 0.7797 | IoU imgs: 12
  📊   20/1000 | 1.63 FPS | mIoU(10): 0.7195 | IoU imgs: 16
  📊   25/1000 | 1.59 FPS | mIoU(10): 0.6951 | IoU imgs: 21
  📊   30/1000 | 1.60 FPS | mIoU(10): 0.8069 | IoU imgs: 24
  📊   35/1000 | 1.57 FPS | mIoU(10): 0.8363 | IoU imgs: 27
  📊   40/1000 | 1.39 FPS | mIoU(10): 0.8294 | IoU imgs: 32
  📊   45/1000 | 1.33 FPS | mIoU(10): 0.7726 | IoU imgs: 37
  📊   50/1000 | 1.29 FPS | mIoU(10): 0.7921 | IoU imgs: 41
  📊   55/1000 | 1.26 FPS | mIoU(10): 0.7814 | IoU imgs: 45
  📊   60/1000 | 1.25 FPS | mIoU(10): 0.7211 | IoU imgs: 49
  📊   65/1000 | 1.21 FPS | mIoU(10): 0.7599 |

## 3.3 Запуск базового пайплайна на изображениях сделанных в плохую видимость

In [39]:
if __name__ == "__main__":
    print("\n=== BASELINE: YOLO26n + EfficientSAM3 (no improvements) ===\n")
    clear_gpu_memory()

    pipeline = YOLO_SAM_Pipeline_Baseline(
        sam_checkpoint='/content/checkpoints/efficient_sam3_efficientvit_s.pt',
        backbone_type='efficientvit',
        model_name='b0',
        confidence_threshold=0.5
    )

    image_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/images/bad'
    mask_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/labels/val'

    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))[:230]
    print(f"Found {len(image_paths)} images")

    gt_mask_paths = []
    for img_path in image_paths:
        img_name = Path(img_path).stem
        mask_found = False
        for suffix in ['_train_color.png', '_train_id.png', '.png']:
            mask_path = os.path.join(mask_dir, f"{img_name}{suffix}")
            if os.path.exists(mask_path):
                gt_mask_paths.append(mask_path)
                mask_found = True
                break
        if not mask_found:
            matching = glob.glob(os.path.join(mask_dir, f"{img_name}*.png"))
            gt_mask_paths.append(matching[0] if matching else None)

    print(f"  Masks found: {sum(1 for p in gt_mask_paths if p)}/{len(image_paths)}")

    metrics = pipeline.process_batch_with_iou(image_paths, gt_mask_paths)
    pipeline.print_iou_summary(metrics)

    # Визуализация
    viz_dir = '/content/viz_results_baseline'
    os.makedirs(viz_dir, exist_ok=True)
    num_viz = min(5, len(image_paths))
    print(f"\n Saving {num_viz} visualizations to {viz_dir}...")

    for i in range(num_viz):
        img_path = image_paths[i]
        gt_path = gt_mask_paths[i]
        masks, classes, scores, img_metrics, yolo_boxes = pipeline._process_single(img_path)
        if masks:
            iou_metrics = None
            if gt_path and os.path.exists(gt_path):
                gt_mask = load_gt_mask(gt_path, Image.open(img_path).size)
                iou_metrics = compute_iou_for_pipeline(masks, classes, gt_mask)
            save_path = os.path.join(viz_dir, f"baseline_{i+1:03d}_{Path(img_path).stem}.png")
            pipeline.visualize_with_iou(img_path, masks, classes, scores, gt_path, iou_metrics, save_path, yolo_boxes)

    print(f" Visualizations saved to: {viz_dir}")
    pipeline.cleanup()
    clear_gpu_memory()


=== BASELINE: YOLO26n + EfficientSAM3 (no improvements) ===

  Device: cuda
 Loading YOLO26n...
 Loading EfficientSAM3...
 Baseline Pipeline ready (conf=0.5)

Found 229 images
  Masks found: 229/229
 Processing 229 images (BASELINE)...
     5/229 | 1.22 FPS | mIoU(10): 0.8799 | IoU imgs: 4
    10/229 | 1.02 FPS | mIoU(10): 0.7945 | IoU imgs: 9
    15/229 | 0.99 FPS | mIoU(10): 0.7244 | IoU imgs: 14
    20/229 | 1.01 FPS | mIoU(10): 0.7016 | IoU imgs: 18
    25/229 | 0.98 FPS | mIoU(10): 0.7323 | IoU imgs: 23
    30/229 | 1.05 FPS | mIoU(10): 0.7541 | IoU imgs: 26
    35/229 | 1.08 FPS | mIoU(10): 0.7798 | IoU imgs: 30
    40/229 | 1.08 FPS | mIoU(10): 0.7602 | IoU imgs: 34
    45/229 | 1.09 FPS | mIoU(10): 0.7042 | IoU imgs: 38
    50/229 | 1.09 FPS | mIoU(10): 0.7419 | IoU imgs: 43
    55/229 | 1.09 FPS | mIoU(10): 0.8007 | IoU imgs: 47
    60/229 | 1.10 FPS | mIoU(10): 0.7799 | IoU imgs: 51
    65/229 | 1.11 FPS | mIoU(10): 0.7473 | IoU imgs: 55
    70/229 | 1.09 FPS | mIoU(10): 0.6

## 3.4 Базовый пайплайн+условия видимости

In [47]:
if __name__ == "__main__":
    print("\n=== YOLO26n + EfficientSAM3 — BDD100K IoU Evaluation ===\n")
    clear_gpu_memory()

    pipeline_improve = YOLO_SAM_Pipeline(
        sam_checkpoint='/content/checkpoints/efficient_sam3_efficientvit_s.pt',
        backbone_type='efficientvit',
        model_name='b0',
        confidence_threshold=0.5
    )

    image_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/images/val'
    mask_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/labels/val'

    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))[:1000]
    print(f"Found {len(image_paths)} images")

    _ = pipeline_improve._process_single(image_paths[0], preprocess=True)
    clear_gpu_memory()

    gt_mask_paths = []
    for img_path in image_paths:
        img_name = Path(img_path).stem
        mask_found = False
        for suffix in ['_train_color.png', '_train_id.png', '.png']:
            mask_path = os.path.join(mask_dir, f"{img_name}{suffix}")
            if os.path.exists(mask_path):
                gt_mask_paths.append(mask_path)
                mask_found = True
                break
        if not mask_found:
            matching = glob.glob(os.path.join(mask_dir, f"{img_name}*.png"))
            gt_mask_paths.append(matching[0] if matching else None)

    print(f"  Masks found: {sum(1 for p in gt_mask_paths if p)}/{len(image_paths)}")

    metrics = pipeline_improve.process_batch_with_iou(image_paths, gt_mask_paths, preprocess=True)
    pipeline_improve.print_iou_summary(metrics)

    # Визуализация нескольких примеров
    viz_dir = '/content/viz_results'
    os.makedirs(viz_dir, exist_ok=True)

    num_viz = min(5, len(image_paths))
    print(f"\nSaving {num_viz} visualizations to {viz_dir}...")

    for i in range(num_viz):
        img_path = image_paths[i]
        gt_path = gt_mask_paths[i]

        # Получаем маски
        masks, classes, scores, img_metrics, yolo_boxes = pipeline_improve._process_single(img_path, preprocess=True)

        if masks:
            # IoU для этого изображения
            iou_metrics = None
            if gt_path and os.path.exists(gt_path):
                gt_mask = load_gt_mask(gt_path, Image.open(img_path).size)
                iou_metrics = compute_iou_for_pipeline(masks, classes, gt_mask)

            save_path = os.path.join(viz_dir, f"viz_{i+1:03d}_{Path(img_path).stem}.png")
            pipeline_improve.visualize_with_iou(img_path, masks, classes, scores, gt_path, iou_metrics, save_path, yolo_boxes)

    print(f" Visualizations saved to: {viz_dir}")

    pipeline_improve.cleanup()
    clear_gpu_memory()


=== YOLO26n + EfficientSAM3 — BDD100K IoU Evaluation ===

  Device: cuda
 Loading YOLO26n (main)...
 Loading YOLO26m (small objects)...
 Loading EfficientSAM3...
 Pipeline ready (conf=0.5)
Found 1000 images
  Masks found: 1000/1000
 Processing 1000 images...
    Adaptive preprocessing ENABLED
      5/1000 | 1.13 FPS | mIoU(10): 0.8338 | IoU imgs: 5
     10/1000 | 0.97 FPS | mIoU(10): 0.8041 | IoU imgs: 10
     15/1000 | 1.08 FPS | mIoU(10): 0.7896 | IoU imgs: 13
     20/1000 | 1.06 FPS | mIoU(10): 0.7204 | IoU imgs: 18
     25/1000 | 1.03 FPS | mIoU(10): 0.7160 | IoU imgs: 23
     30/1000 | 1.07 FPS | mIoU(10): 0.7978 | IoU imgs: 27
     35/1000 | 1.07 FPS | mIoU(10): 0.8306 | IoU imgs: 32
     40/1000 | 1.02 FPS | mIoU(10): 0.7774 | IoU imgs: 37
     45/1000 | 1.03 FPS | mIoU(10): 0.7196 | IoU imgs: 42
     50/1000 | 1.02 FPS | mIoU(10): 0.7994 | IoU imgs: 47
     55/1000 | 1.00 FPS | mIoU(10): 0.7770 | IoU imgs: 52
     60/1000 | 1.00 FPS | mIoU(10): 0.7563 | IoU imgs: 57
     65/10

In [48]:
if __name__ == "__main__":
    print("\n=== YOLO26n + EfficientSAM3 — BDD100K IoU Evaluation ===\n")
    clear_gpu_memory()

    pipeline_improve = YOLO_SAM_Pipeline(
        sam_checkpoint='/content/checkpoints/efficient_sam3_efficientvit_s.pt',
        backbone_type='efficientvit',
        model_name='b0',
        confidence_threshold=0.5
    )

    image_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/images/bad'
    mask_dir = '/content/drive/MyDrive/sam3_bdd_project/data_seg/labels/val'

    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))[:230]
    print(f"Found {len(image_paths)} images")

    _ = pipeline_improve._process_single(image_paths[0], preprocess=True)
    clear_gpu_memory()

    gt_mask_paths = []
    for img_path in image_paths:
        img_name = Path(img_path).stem
        mask_found = False
        for suffix in ['_train_color.png', '_train_id.png', '.png']:
            mask_path = os.path.join(mask_dir, f"{img_name}{suffix}")
            if os.path.exists(mask_path):
                gt_mask_paths.append(mask_path)
                mask_found = True
                break
        if not mask_found:
            matching = glob.glob(os.path.join(mask_dir, f"{img_name}*.png"))
            gt_mask_paths.append(matching[0] if matching else None)

    print(f"  Masks found: {sum(1 for p in gt_mask_paths if p)}/{len(image_paths)}")

    metrics = pipeline_improve.process_batch_with_iou(image_paths, gt_mask_paths, preprocess=True)
    pipeline_improve.print_iou_summary(metrics)

    # Визуализация нескольких примеров
    viz_dir = '/content/viz_results'
    os.makedirs(viz_dir, exist_ok=True)

    num_viz = min(5, len(image_paths))
    print(f"\nSaving {num_viz} visualizations to {viz_dir}...")

    for i in range(num_viz):
        img_path = image_paths[i]
        gt_path = gt_mask_paths[i]

        # Получаем маски
        masks, classes, scores, img_metrics, yolo_boxes = pipeline_improve._process_single(img_path, preprocess=True)

        if masks:
            # IoU для этого изображения
            iou_metrics = None
            if gt_path and os.path.exists(gt_path):
                gt_mask = load_gt_mask(gt_path, Image.open(img_path).size)
                iou_metrics = compute_iou_for_pipeline(masks, classes, gt_mask)

            save_path = os.path.join(viz_dir, f"viz_{i+1:03d}_{Path(img_path).stem}.png")
            pipeline_improve.visualize_with_iou(img_path, masks, classes, scores, gt_path, iou_metrics, save_path, yolo_boxes)

    print(f" Visualizations saved to: {viz_dir}")

    pipeline_improve.cleanup()
    clear_gpu_memory()


=== YOLO26n + EfficientSAM3 — BDD100K IoU Evaluation ===

  Device: cuda
 Loading YOLO26n (main)...
 Loading YOLO26m (small objects)...
 Loading EfficientSAM3...
 Pipeline ready (conf=0.5)
Found 229 images
  Masks found: 229/229
 Processing 229 images...
    Adaptive preprocessing ENABLED
      5/229 | 1.13 FPS | mIoU(10): 0.8901 | IoU imgs: 4
     10/229 | 1.02 FPS | mIoU(10): 0.7347 | IoU imgs: 9
     15/229 | 0.93 FPS | mIoU(10): 0.6782 | IoU imgs: 14
     20/229 | 0.93 FPS | mIoU(10): 0.7294 | IoU imgs: 19
     25/229 | 0.92 FPS | mIoU(10): 0.7463 | IoU imgs: 24
     30/229 | 0.95 FPS | mIoU(10): 0.7623 | IoU imgs: 27
     35/229 | 0.95 FPS | mIoU(10): 0.7672 | IoU imgs: 32
     40/229 | 0.92 FPS | mIoU(10): 0.7672 | IoU imgs: 37
     45/229 | 0.92 FPS | mIoU(10): 0.6862 | IoU imgs: 42
     50/229 | 0.92 FPS | mIoU(10): 0.7026 | IoU imgs: 47
     55/229 | 0.91 FPS | mIoU(10): 0.7747 | IoU imgs: 52
     60/229 | 0.91 FPS | mIoU(10): 0.7789 | IoU imgs: 57
     65/229 | 0.90 FPS | mI